This notebook adds:

* BM25 keyword search
* FAISS semantic search
* Score fusion
* Better retrieval quality

### Install BM25 (run once)

In [1]:
!pip install rank-bm25

### Imports

In [2]:
import pickle
import numpy as np
import faiss

from rank_bm25 import BM25Okapi

### Load Metadata

In [3]:
with open("data/metadata.pkl", "rb") as f:
    metadata = pickle.load(f)

print("Total chunks:", len(metadata))

Total chunks: 6


### Prepare Text Documents

In [4]:
documents = []

for item in metadata:
    documents.append(item["text"])

print(documents[:2])

['Cloud Migration Guide\n\nAWS is the primary cloud platform.\n\nKubernetes is used for deployment.\n\nGitHub Actions manages CI/CD.\n\nPrometheus and Grafana are used for monitoring.', 'Company Policies\n\nPasswords must be changed every 90 days.\n\nVPN access is mandatory.\n\nMulti-factor authentication is required.\n\nConfidential data should not be shared externally.']


### Build BM25 Index

In [5]:
tokenized_docs = [
    doc.lower().split()
    for doc in documents
]

bm25 = BM25Okapi(tokenized_docs)

print("BM25 index created successfully!")

BM25 index created successfully!


### Save BM25

In [6]:
with open("data/bm25.pkl", "wb") as f:
    pickle.dump(bm25, f)

print("BM25 saved successfully!")

BM25 saved successfully!


### Load FAISS Components

In [7]:
index = faiss.read_index("data/faiss_index.bin")

with open("data/vectorizer.pkl", "rb") as f:
    vectorizer = pickle.load(f)

print("FAISS loaded successfully!")

FAISS loaded successfully!


### FAISS Search Function

In [8]:
def faiss_search(question, top_k=3):

    query_vector = vectorizer.transform([question]).toarray().astype("float32")

    distances, indices = index.search(query_vector, top_k)

    results = []

    for idx in indices[0]:

        results.append({
            "text": metadata[idx]["text"],
            "source": metadata[idx]["source"]
        })

    return results

### BM25 Search Function

In [9]:
def bm25_search(question, top_k=3):

    tokens = question.lower().split()

    scores = bm25.get_scores(tokens)

    top_indices = np.argsort(scores)[::-1][:top_k]

    results = []

    for idx in top_indices:

        results.append({
            "text": metadata[idx]["text"],
            "source": metadata[idx]["source"]
        })

    return results

### Hybrid Search

In [10]:
def hybrid_search(question):

    faiss_results = faiss_search(question)
    bm25_results = bm25_search(question)

    combined = faiss_results + bm25_results

    seen = set()
    final_results = []

    for item in combined:

        if item["source"] not in seen:

            seen.add(item["source"])
            final_results.append(item)

    return final_results

### Test Hybrid Search

In [11]:
question = "How many leave days are allowed?"

results = hybrid_search(question)

for i, item in enumerate(results, start=1):

    print("=" * 70)
    print("Result", i)
    print("Source:", item["source"])
    print()
    print(item["text"])

Result 1
Source: employee_handbook.txt

Employee Handbook

Employees receive 20 annual leave days.

Remote work is allowed three days per week.

Working hours are 9 AM to 6 PM.

Health insurance benefits are provided.
Result 2
Source: company_policies.txt

Company Policies

Passwords must be changed every 90 days.

VPN access is mandatory.

Multi-factor authentication is required.

Confidential data should not be shared externally.
Result 3
Source: cloud_migration_guide.txt

Cloud Migration Guide

AWS is the primary cloud platform.

Kubernetes is used for deployment.

GitHub Actions manages CI/CD.

Prometheus and Grafana are used for monitoring.
Result 4
Source: products.csv

ProductID   Product     Category  Price
0      P101    Laptop  Electronics    800
1      P102     Mouse  Electronics     25
2      P103  Keyboard  Electronics     60


###  Test Another Query

In [12]:
question = "What monitoring tools are used?"

results = hybrid_search(question)

for item in results:

    print(item["source"])

cloud_migration_guide.txt
employee_handbook.txt
company_policies.txt
products.csv


### Save Hybrid Components

In [13]:
with open("data/bm25.pkl", "wb") as f:
    pickle.dump(bm25, f)

print("Notebook 8 completed successfully!")

Notebook 8 completed successfully!
